In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("=== TAHAP 1: DATA GENERATION (SINTETIS CITRA SATELIT) ===")
np.random.seed(42)
tf.random.set_seed(42)

jumlah_data = 1000
ukuran_citra = 64 # 64x64 piksel

# Menggenerate citra dengan bias warna agar CNN bisa belajar polanya
X = np.zeros((jumlah_data, ukuran_citra, ukuran_citra, 3))
y = np.zeros(jumlah_data)

# Skenario Imbalance: 85% Kawasan Hijau (Aman), 15% Deforestasi (Kritis)
n_deforestasi = int(0.15 * jumlah_data)
indeks_deforestasi = np.random.choice(jumlah_data, n_deforestasi, replace=False)

for i in range(jumlah_data):
    if i in indeks_deforestasi:
        # Citra Deforestasi: Dominan merah/coklat (Channel R dan G tinggi, B rendah)
        X[i] = np.random.rand(ukuran_citra, ukuran_citra, 3) * 0.5 + [0.4, 0.3, 0.0]
        y[i] = 1 # Label 1 = Deforestasi
    else:
        # Citra Hijau: Dominan warna hijau (Channel G tinggi)
        X[i] = np.random.rand(ukuran_citra, ukuran_citra, 3) * 0.5 + [0.1, 0.5, 0.1]
        y[i] = 0 # Label 0 = Aman

# Normalisasi Piksel (0 - 1)
X = np.clip(X, 0.0, 1.0)

# Memisahkan Data Latih dan Uji (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Distribusi Data Latih: {len(y_train)} citra.")
print(f"Distribusi Kelas Latih: Aman (0) = {np.sum(y_train == 0)}, Deforestasi (1) = {np.sum(y_train == 1)}")


print("\n=== TAHAP 2: ARSITEKTUR CONVOLUTIONAL NEURAL NETWORK (CNN) ===")
# Membangun arsitektur ekstraksi fitur spasial
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(ukuran_citra, ukuran_citra, 3)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3), # Mencegah overfitting
    layers.Dense(1, activation='sigmoid') # Sigmoid untuk output biner
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


print("\n=== TAHAP 3: PELATIHAN MODEL & PENANGANAN IMBALANCE ===")
# Memberikan bobot lebih berat pada kelas minoritas (Deforestasi) agar model tidak mengabaikannya
bobot_kelas = {
    0: 1.0, 
    1: (len(y_train) / (2 * np.sum(y_train == 1))) # Bobot lebih besar untuk Kelas 1
}

# Melatih Model
history = model.fit(
    X_train, y_train, 
    epochs=10, 
    batch_size=32,
    validation_split=0.2,
    class_weight=bobot_kelas,
    verbose=1
)


print("\n=== TAHAP 4: EVALUASI & VISUALISASI METRIK BISNIS ===")
# Prediksi data uji
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Menampilkan Laporan Metrik
print("\n--- Laporan Klasifikasi (Fokus pada Recall Kelas 1) ---")
print(classification_report(y_test, y_pred, target_names=['0: Kawasan Hijau', '1: Deforestasi']))

# Memvisualisasikan Kurva Pelatihan dan Confusion Matrix
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Kurva Akurasi
ax[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
ax[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
ax[0].set_title('Kurva Pembelajaran (Accuracy)', fontweight='bold')
ax[0].set_xlabel('Epochs')
ax[0].legend()

# Plot 2: Kurva Loss
ax[1].plot(history.history['loss'], label='Train Loss', color='blue')
ax[1].plot(history.history['val_loss'], label='Val Loss', color='orange')
ax[1].set_title('Kurva Kesalahan (Loss)', fontweight='bold')
ax[1].set_xlabel('Epochs')
ax[1].legend()

# Plot 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax[2], cbar=False,
            xticklabels=['Hijau (Pred)', 'Deforestasi (Pred)'],
            yticklabels=['Hijau (Aktual)', 'Deforestasi (Aktual)'])
ax[2].set_title('Confusion Matrix\n(Minimalisir False Negative)', fontweight='bold')

plt.tight_layout()
plt.show()